In [ ]:
# -*- coding: utf-8 -*-
"""
Код для запуска в Google Colab (v3 - Separate Scalers).
Каждая модель (RF, MLP) обучается со своим скейлером.
Копирует данные с Диска в Colab перед обучением.
"""

# Установка
!pip install skl2onnx onnxruntime scikit-learn pandas numpy

import os
import json
import zipfile
import shutil
import numpy as np
import pandas as pd
from glob import glob
from pathlib import Path

# Подключение Google Drive
from google.colab import drive
drive.mount('/content/drive')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# ============== НАСТРОЙКИ ==============

# Путь на Google Drive
DRIVE_SOURCE_DIR = "/content/drive/MyDrive/diploma_data"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/diploma_models"

# Локальные пути в Colab (быстрый диск, очищается после закрытия)
LOCAL_DATA_DIR = "/content/data"
LOCAL_MODELS_DIR = "/content/models"

FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    'Fwd Packet Length Mean', 'Bwd Packet Length Mean', 'Flow Packets/s',
]

MAX_ROWS_PER_FILE = 500_000
SAMPLE_FRACTION = 0.2

# ============== ФУНКЦИИ ==============

def copy_data_locally():
    print(f"🚀 Копирование данных с Google Drive в локальную среду Colab...")
    if not os.path.exists(LOCAL_DATA_DIR):
        os.makedirs(LOCAL_DATA_DIR)

    zip_files = glob(f"{DRIVE_SOURCE_DIR}/*.zip")
    if not zip_files:
        raise FileNotFoundError(f"ZIP файлы не найдены в {DRIVE_SOURCE_DIR}")

    local_zips = []
    for zf in zip_files:
        filename = Path(zf).name
        dest = f"{LOCAL_DATA_DIR}/{filename}"
        print(f"   Копирую {filename}...")
        shutil.copy(zf, dest)
        local_zips.append(dest)

    print("✅ Копирование завершено.")
    return local_zips

def load_data(zip_files):
    dfs = []
    print(f"\n📦 Чтение данных из локальных копий...")

    for zpath in zip_files:
        print(f"   Архив: {Path(zpath).name}")
        with zipfile.ZipFile(zpath, 'r') as zf:
            csvs = [f for f in zf.namelist() if f.endswith('.csv')]
            for csv_file in csvs:
                with zf.open(csv_file) as f:
                    try:
                        for chunk in pd.read_csv(f, usecols=lambda x: x.strip() in FEATURES + [' Label', 'Label'],
                                               chunksize=100_000, low_memory=False):
                            chunk.columns = chunk.columns.str.strip()
                            if SAMPLE_FRACTION < 1.0:
                                chunk = chunk.sample(frac=SAMPLE_FRACTION, random_state=42)
                            dfs.append(chunk)
                            if len(chunk) >= MAX_ROWS_PER_FILE: break
                    except Exception as e:
                        print(f"   ⚠️ Ошибка чтения {csv_file}: {e}")

    return pd.concat(dfs, ignore_index=True)


def save_scaler_params(scaler, features, path):
    """Сохраняет параметры скейлера в JSON"""
    params = {
        "features": features,
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist()
    }
    with open(path, 'w') as f:
        json.dump(params, f, indent=2)
    print(f"   ✅ Скейлер сохранён: {path}")


def main():
    # 1. Подготовка
    os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

    # 2. Копирование и загрузка
    local_zips = copy_data_locally()
    df = load_data(local_zips)
    print(f"✅ Всего строк загружено: {len(df)}")

    # 3. Препроцессинг
    label_col = [c for c in df.columns if 'Label' in c][0]
    X = df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = (df[label_col].str.strip().str.upper() != 'BENIGN').astype(int)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 4. Обучение с РАЗДЕЛЬНЫМИ скейлерами

    # === Random Forest: свой скейлер ===
    print("\n📊 Нормализация данных для RF (StandardScaler)...")
    rf_scaler = StandardScaler()
    X_train_rf = rf_scaler.fit_transform(X_train)
    X_test_rf = rf_scaler.transform(X_test)

    print("🌲 Обучение Random Forest...")
    rf = RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
    rf.fit(X_train_rf, y_train)
    print(f"   Accuracy: {accuracy_score(y_test, rf.predict(X_test_rf)):.4f}")

    # === MLP: свой скейлер ===
    print("\n📊 Нормализация данных для MLP (StandardScaler)...")
    mlp_scaler = StandardScaler()
    X_train_mlp = mlp_scaler.fit_transform(X_train)
    X_test_mlp = mlp_scaler.transform(X_test)

    print("🧠 Обучение MLP...")
    mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=200, random_state=42)
    mlp.fit(X_train_mlp, y_train)
    print(f"   Accuracy: {accuracy_score(y_test, mlp.predict(X_test_mlp)):.4f}")

    # 5. Сохранение скейлеров
    print("\n💾 Сохранение скейлеров...")
    save_scaler_params(rf_scaler, FEATURES, f"{LOCAL_MODELS_DIR}/rf_scaler_params.json")
    save_scaler_params(mlp_scaler, FEATURES, f"{LOCAL_MODELS_DIR}/mlp_scaler_params.json")
    # Обратная совместимость
    save_scaler_params(rf_scaler, FEATURES, f"{LOCAL_MODELS_DIR}/scaler_params.json")

    # 6. Экспорт ONNX и копирование на Drive
    print("\n💾 Экспорт моделей и копирование на Google Drive...")
    for name, model in [("rf_model", rf), ("mlp_model", mlp)]:
        onnx_model = convert_sklearn(model, initial_types=[('features', FloatTensorType([None, len(FEATURES)]))])
        local_path = f"{LOCAL_MODELS_DIR}/{name}.onnx"
        with open(local_path, "wb") as f:
            f.write(onnx_model.SerializeToString())
        shutil.copy(local_path, f"{DRIVE_OUTPUT_DIR}/{name}.onnx")
        print(f"   ✅ {name}.onnx")

    # Копируем скейлеры на Drive
    for scaler_file in ["rf_scaler_params.json", "mlp_scaler_params.json", "scaler_params.json"]:
        shutil.copy(f"{LOCAL_MODELS_DIR}/{scaler_file}", f"{DRIVE_OUTPUT_DIR}/{scaler_file}")

    print(f"\n✅ УСПЕХ! Модели и скейлеры сохранены в: {DRIVE_OUTPUT_DIR}")
    print(f"   📁 rf_model.onnx + rf_scaler_params.json")
    print(f"   📁 mlp_model.onnx + mlp_scaler_params.json")
    print(f"   📁 scaler_params.json (обратная совместимость)")

if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 52.5 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Копирование данных с Google Drive в локальную среду Colab...
   Копирую CSV-03-11.zip...
   Копирую CSV-01-12.zip...
✅ Копирование завершено.

📦 Чтение данных из локальных копий...
   Архив: CSV-03-11.zip
   Архив: CSV-01-12.zip
✅ Всего строк загружено: 14085526

📊 Нормализация данных для RF (StandardScaler)...
🌲 Обучение Random Forest...
   Accuracy: 0.9997

📊 Нормализация данных для MLP (StandardScaler)...
🧠 Обучение MLP...
   Accuracy: 0.9995

💾 Сохранение скейлеров...
   ✅ Скейлер сохранён: /content/models/rf_scaler_params.json
   ✅ Скейлер сохранён: /content/models/mlp_scaler_params.json
   ✅ Скейлер сохранён: /content

In [1]:
# -*- coding: utf-8 -*-
"""
Код для запуска в Google Colab (v3 - Separate Scalers).
Каждая модель (RF, MLP) обучается со своим скейлером.
Копирует данные с Диска в Colab перед обучением.
"""

# Установка
!pip install skl2onnx onnxruntime scikit-learn pandas numpy

import os
import json
import zipfile
import shutil
import numpy as np
import pandas as pd
from glob import glob
from pathlib import Path

# Подключение Google Drive
from google.colab import drive
drive.mount('/content/drive')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# ============== НАСТРОЙКИ ==============

# Путь на Google Drive
DRIVE_SOURCE_DIR = "/content/drive/MyDrive/diploma_data"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/diploma_models"

# Локальные пути в Colab (быстрый диск, очищается после закрытия)
LOCAL_DATA_DIR = "/content/data"
LOCAL_MODELS_DIR = "/content/models"

FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    'Fwd Packet Length Mean', 'Bwd Packet Length Mean', 'Flow Packets/s',
]

MAX_ROWS_PER_FILE = 500_000
SAMPLE_FRACTION = 0.2

# ============== ФУНКЦИИ ==============

def copy_data_locally():
    print(f"🚀 Копирование данных с Google Drive в локальную среду Colab...")
    if not os.path.exists(LOCAL_DATA_DIR):
        os.makedirs(LOCAL_DATA_DIR)

    zip_files = glob(f"{DRIVE_SOURCE_DIR}/*.zip")
    if not zip_files:
        raise FileNotFoundError(f"ZIP файлы не найдены в {DRIVE_SOURCE_DIR}")

    local_zips = []
    for zf in zip_files:
        filename = Path(zf).name
        dest = f"{LOCAL_DATA_DIR}/{filename}"
        print(f"   Копирую {filename}...")
        shutil.copy(zf, dest)
        local_zips.append(dest)

    print("✅ Копирование завершено.")
    return local_zips

def load_data(zip_files):
    dfs = []
    print(f"\n📦 Чтение данных из локальных копий...")

    for zpath in zip_files:
        print(f"   Архив: {Path(zpath).name}")
        with zipfile.ZipFile(zpath, 'r') as zf:
            csvs = [f for f in zf.namelist() if f.endswith('.csv')]
            for csv_file in csvs:
                with zf.open(csv_file) as f:
                    try:
                        for chunk in pd.read_csv(f, usecols=lambda x: x.strip() in FEATURES + [' Label', 'Label'],
                                               chunksize=100_000, low_memory=False):
                            chunk.columns = chunk.columns.str.strip()
                            if SAMPLE_FRACTION < 1.0:
                                chunk = chunk.sample(frac=SAMPLE_FRACTION, random_state=42)
                            dfs.append(chunk)
                            if len(chunk) >= MAX_ROWS_PER_FILE: break
                    except Exception as e:
                        print(f"   ⚠️ Ошибка чтения {csv_file}: {e}")

    return pd.concat(dfs, ignore_index=True)


def save_scaler_params(scaler, features, path, use_log1p=False):
    """Сохраняет параметры скейлера в JSON"""
    params = {
        "features": features,
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
        "use_log1p": use_log1p
    }
    with open(path, 'w') as f:
        json.dump(params, f, indent=2)
    print(f"   ✅ Скейлер сохранён: {path} (use_log1p={use_log1p})")


def main():
    # 1. Подготовка
    os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

    # 2. Копирование и загрузка
    local_zips = copy_data_locally()
    df = load_data(local_zips)
    print(f"✅ Всего строк загружено: {len(df)}")

    # 3. Препроцессинг
    label_col = [c for c in df.columns if 'Label' in c][0]
    X = df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = (df[label_col].str.strip().str.upper() != 'BENIGN').astype(int)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 4. Обучение с РАЗДЕЛЬНЫМИ скейлерами и логарифмированием (Log1p)

    print("\n📈 Применение логарифмирования (Log1p) для защиты от аномальных выбросов...")
    X_train_log = np.log1p(X_train)
    X_test_log = np.log1p(X_test)

    # === Random Forest: обычный скейлер (без логарифмирования) ===
    print("\n📊 Нормализация данных для RF (StandardScaler)...")
    rf_scaler = StandardScaler()
    X_train_rf = rf_scaler.fit_transform(X_train)
    X_test_rf = rf_scaler.transform(X_test)

    print("🌲 Обучение Random Forest...")
    rf = RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
    rf.fit(X_train_rf, y_train)
    print(f"   Accuracy: {accuracy_score(y_test, rf.predict(X_test_rf)):.4f}")

    # === MLP: свой скейлер ===
    print("\n📊 Нормализация данных для MLP (StandardScaler)...")
    mlp_scaler = StandardScaler()
    X_train_mlp = mlp_scaler.fit_transform(X_train_log)
    X_test_mlp = mlp_scaler.transform(X_test_log)

    print("🧠 Обучение MLP...")
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        max_iter=500,
        early_stopping=True,
        random_state=42
    )
    mlp.fit(X_train_mlp, y_train)
    print(f"   Accuracy: {accuracy_score(y_test, mlp.predict(X_test_mlp)):.4f}")

    # 5. Сохранение скейлеров с указанием флага логарифмирования
    print("\n💾 Сохранение скейлеров...")
    save_scaler_params(rf_scaler, FEATURES, f"{LOCAL_MODELS_DIR}/rf_scaler_params.json", use_log1p=False)
    save_scaler_params(mlp_scaler, FEATURES, f"{LOCAL_MODELS_DIR}/mlp_scaler_params.json", use_log1p=True)
    # Обратная совместимость (копия RF)
    save_scaler_params(rf_scaler, FEATURES, f"{LOCAL_MODELS_DIR}/scaler_params.json", use_log1p=False)

    # 6. Экспорт ONNX и копирование на Drive
    print("\n💾 Экспорт моделей и копирование на Google Drive...")
    for name, model in [("rf_model", rf), ("mlp_model", mlp)]:
        onnx_model = convert_sklearn(model, initial_types=[('features', FloatTensorType([None, len(FEATURES)]))])
        local_path = f"{LOCAL_MODELS_DIR}/{name}.onnx"
        with open(local_path, "wb") as f:
            f.write(onnx_model.SerializeToString())
        shutil.copy(local_path, f"{DRIVE_OUTPUT_DIR}/{name}.onnx")
        print(f"   ✅ {name}.onnx")

    # Копируем скейлеры на Drive
    for scaler_file in ["rf_scaler_params.json", "mlp_scaler_params.json", "scaler_params.json"]:
        shutil.copy(f"{LOCAL_MODELS_DIR}/{scaler_file}", f"{DRIVE_OUTPUT_DIR}/{scaler_file}")

    print(f"\n✅ УСПЕХ! Модели и скейлеры сохранены в: {DRIVE_OUTPUT_DIR}")
    print(f"   📁 rf_model.onnx + rf_scaler_params.json")
    print(f"   📁 mlp_model.onnx + mlp_scaler_params.json")
    print(f"   📁 scaler_params.json (обратная совместимость)")

if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 44.7 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Копирование данных с Google Drive в локальную среду Colab...
   Копирую CSV-03-11.zip...
   Копирую CSV-01-12.zip...
✅ Копирование завершено.

📦 Чтение данных из локальных копий...
   Архив: CSV-03-11.zip
   Архив: CSV-01-12.zip
✅ Всего строк загружено: 14085526

📈 Применение логарифмирования (Log1p) для защиты от аномальных выбросов...

📊 Нормализация данных для RF (StandardScaler)...
🌲 Обучение Random Forest...
   Accuracy: 0.9997

📊 Нормализация данных для MLP (StandardScaler)...
🧠 Обучение MLP...
   Accuracy: 0.9997

💾 Сохранение скейлеров...
   ✅ Скейлер сохранён: /content/models/rf_scaler_params.json (use_log1p=False)

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive
